# LaStBeRu × Rubin reference-catalog analysis

This notebook runs or inspects the large-catalog workflow. It counts individual detector images by band using a single uploaded-table TAP cross-match per chunk, samples coadd property maps at every coordinate, ranks cutout candidates, and displays representative 20-arcsec cutouts.

The first complete 35k-target run can take substantial time. TAP chunks are cached, so an interrupted run resumes without repeating completed queries.

## 1. Configuration

In [ ]:
from pathlib import Path
from dataclasses import replace
import json
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

from target_selection import (
    load_reference_catalog_config, run_reference_catalog,
    run_reference_catalog_preview,
)

CONFIG_PATH = Path("configs/lastberu_dp2.toml")
RUN_CATALOG = False  # False only inspects an existing dataset.
TRIAL_TARGET_LIMIT = 200  # Set None for all 35,400 targets.
TRIAL_MAX_CUTOUTS = 12
DISPLAY_CUTOUTS = 12
config = load_reference_catalog_config(CONFIG_PATH)
config


## 2. Run or load the dataset

For a quick test, keep `TRIAL_TARGET_LIMIT=200`; the preview helper uses the same cache and spatial-tile parallelism as the complete run. Set `TRIAL_TARGET_LIMIT=None` to process all targets. The full run is intentionally not executed by this notebook unless `RUN_CATALOG=True`.


In [ ]:
if RUN_CATALOG:
    if TRIAL_TARGET_LIMIT is None:
        catalog_result = run_reference_catalog(config)
    else:
        catalog_result = run_reference_catalog_preview(
            config, n_targets=TRIAL_TARGET_LIMIT, max_cutouts=TRIAL_MAX_CUTOUTS
        )
    catalog = catalog_result.catalog
    output_directory = catalog_result.output_dir
    summary = dict(catalog_result.summary)
else:
    output_directory = Path(config.output_dir) / config.name
    catalog_path = output_directory / "reference_catalog.csv"
    manifest_path = output_directory / "manifest.json"
    if not catalog_path.exists():
        raise FileNotFoundError(
            f"No existing dataset at {catalog_path}. Set RUN_CATALOG=True and rerun this cell."
        )
    catalog = pd.read_csv(catalog_path)
    summary = json.loads(manifest_path.read_text())["summary"]

display(pd.Series(summary, name="value").to_frame())
print(f"Products: {output_directory.resolve()}")


## 3. Coverage and grade overview

In [ ]:
coverage_by_grade = pd.crosstab(
    catalog["selection_grade"].fillna("unavailable"),
    catalog["has_coadd"].fillna(False),
    margins=True,
)
coverage_by_grade.columns = ["no coadd", "has coadd", "total"][:len(coverage_by_grade.columns)]
display(coverage_by_grade)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
catalog["n_images_total"].clip(upper=catalog["n_images_total"].quantile(0.99)).hist(ax=axes[0], bins=35)
axes[0].set(title="Individual-image coverage", xlabel="Images per target (clipped at 99th percentile)", ylabel="Targets")
catalog.loc[catalog["has_coadd"], "coadd_quality_score"].hist(ax=axes[1], bins=30)
axes[1].set(title="Coadd quality ranking", xlabel="Quality score", ylabel="Targets")
fig.tight_layout()
plt.show()

## 4. Highest-priority targets

Grade is the primary ordering key (`A`, then `B`). The quality score orders targets only within a grade and combines coadd depth, PSF FWHM, and available-band fraction.

In [ ]:
priority_columns = [
    "cutout_priority_rank", "target_id", "selection_grade",
    "coadd_quality_score", "coadd_n_bands", "coadd_bands",
    "n_images_total", "coadd_maglim_median",
    "coadd_psf_fwhm_median_arcsec", "cutout_status",
]
priority = catalog.loc[catalog["cutout_eligible"]].sort_values("cutout_priority_rank")
display(priority[priority_columns].head(30).style.format({
    "coadd_quality_score": "{:.1f}",
    "coadd_maglim_median": "{:.2f}",
    "coadd_psf_fwhm_median_arcsec": "{:.2f}",
}))

## 5. Generated 20-arcsec cutout grids

In [ ]:
generated = catalog.loc[catalog["cutout_status"].eq("generated")].sort_values("cutout_priority_rank")
print(f"Generated cutouts: {len(generated)}; displaying: {min(DISPLAY_CUTOUTS, len(generated))}")
for row in generated.head(DISPLAY_CUTOUTS).itertuples(index=False):
    print(f"Rank {int(row.cutout_priority_rank)} — {row.target_id} — grade {row.selection_grade}")
    display(Image(filename=row.cutout_path))